In [7]:
import json
import os
import pandas as pd

In [8]:
model_to_configs = {
    'Llama-3.1-8B-Instruct': [
        'full', '0-10000-64', '0-16-64', '64-16-64', '64-16-128'
    ],
    'Mistral-7B-Instruct-v0.2': [
        'full', '0-10000-64', '0-16-64', '64-16-64', '64-16-128'
    ]
}

results_base = 'LEval/results'

In [17]:
for model in model_to_configs:
    df_final = None

    for config_name in model_to_configs[model]:
        if config_name == 'full':
            file_name = f'{model}-no-flexicache'
        else:
            config = config_name.split('-')
            unstable = int(config[0])
            rerank = int(config[1])
            topK = int(config[2])
            file_name = f'{model}-{unstable}-unstable-{rerank}-rerank-topK-{topK}'
        
        file_path = os.path.join(results_base, f'{file_name}.jsonl')

        with open(file_path, 'r') as f:
            data = [json.loads(line) for line in f]

        df = pd.DataFrame(data)
        df = df.rename(columns={'task': 'Task', 'LEval_score': config_name})

        # merge into final DataFrame
        if df_final is None:
            df_final = df
        else:
            df_final = df_final.merge(df, on="Task", how="outer")

    df_final = df_final.set_index("Task")
    df_final.to_csv(f'{model}_LEval_results.csv')
    
    print(model)
    print(df_final)
    print('-' * 100)

Llama-3.1-8B-Instruct
                      full  0-10000-64  0-16-64  64-16-64  64-16-128
Task                                                                
financial_qa       49.6125     43.2787  49.0466   51.9025    50.7827
gov_report_summ    27.1868     26.7441  26.1791   28.8939    26.2449
legal_contract_qa  37.1995     22.6278  29.3843   37.7392    38.0665
meeting_summ       17.9486     16.8507  17.7176   17.5269    17.3808
news_summ          17.4462     17.2711  17.1394   17.6945    17.9532
paper_assistant    21.1826     19.2186  20.4005   20.9741    20.7762
patent_summ        33.6085     23.1207  30.9169   31.6863    33.0350
review_summ        16.7587     16.3966  16.7122   16.7480    16.4838
tv_show_summ       15.4576     15.5752  16.5225   13.7936    14.4966
----------------------------------------------------------------------------------------------------
Mistral-7B-Instruct-v0.2
                      full  0-10000-64  0-16-64  64-16-64  64-16-128
Task                    